# esankhyiki - Examples

Python client for India's National Statistical Office (MoSPI) data portal.
Access indicators across 22 datasets: employment, prices, GDP, health, education, environment, and more.

```bash
pip install mospi-esankhyiki[pandas]
```

In [1]:
import warnings
warnings.filterwarnings('ignore')
import esankhyiki
from esankhyiki.exceptions import InvalidDatasetError, InvalidFilterError, APIError, NoDataError

---
## Available Datasets

In [ ]:
esankhyiki.list_datasets(format="df")

---
## PLFS - Periodic Labour Force Survey

Covers labour force participation, unemployment, worker population ratio, wages.
Use `get_indicators` to see what indicator codes are available, `get_metadata` to see valid filter values for a given indicator.

In [ ]:
# Indicators grouped by frequency_code
indicators = esankhyiki.get_indicators("PLFS")
for group, items in indicators["indicators_by_frequency"].items():
    print(f"\n{group}:")
    for item in items:
        print(f"  {item['indicator_code']}: {item.get('indicator_name', item.get('description', ''))}")

In [ ]:
# Valid filter values for Unemployment Rate (indicator_code=3), Annual (frequency_code=1)
metadata = esankhyiki.get_metadata("PLFS", indicator_code=3, frequency_code=1, year_type_code=1)
for key, values in metadata["filter_values"]["data"].items():
    print(f"\n{key}:")
    for v in values[:4]:
        print(f"  {v}")

In [ ]:
# Unemployment Rate - All India, all years, both sectors
esankhyiki.get_data("PLFS", {
    "indicator_code": 3,
    "frequency_code": 1,
    "year_type_code":1,
    "state_code": 99,     # 99 = All India
    "gender_code": 3,     # 3 = Person
    "age_code": 1,        # 1 = 15 years and above
    "sector_code": 3,     # 3 = rural + urban
}, format="df")

---
## CPI - Consumer Price Index

Multiple base years (2010, 2012, 2024) and two levels: Group and Item.

In [ ]:
# Available base years, levels, and series
esankhyiki.get_indicators("CPI", format="df")

In [ ]:
# Filter values for base_year=2012, Group level
metadata = esankhyiki.get_metadata("CPI", base_year="2012", level="Group", series="Current", format="df")
metadata

In [ ]:
# CPI Group index, base year 2012, year 2024
esankhyiki.get_data("CPI", {
    "base_year": "2012",
    "series": "Current",
    "year": "2024",
}, format="df")

---
## NAS - National Accounts Statistics (GDP)

Annual and quarterly GDP, GVA, consumption, capital formation by industry and institutional sector.

In [ ]:
esankhyiki.get_indicators("NAS", format="df")

In [ ]:
# Filter values for GVA by industry (indicator_code=1), base year 2022-23
metadata = esankhyiki.get_metadata("NAS", indicator_code=1, base_year="2022-23", series="Current", frequency_code=1)
for key, values in metadata.items():
    if isinstance(values, list):
        print(f"{key}: {values[:3]}")

In [ ]:
# GVA by industry, current prices, latest estimates
esankhyiki.get_data("NAS", {
    "indicator_code": 1,
    "base_year": "2022-23",
    "series": "Current",
    "frequency_code": 1,
    "year": "2024-25",
}, format="df")

---
## IIP - Index of Industrial Production

Monthly and annual industrial output across manufacturing, mining, and electricity sectors.

In [ ]:
# Available categories and years for IIP annual
metadata = esankhyiki.get_metadata("IIP", base_year="2011-12", frequency="Annually")
metadata

In [ ]:
# IIP annual, 2023-24
esankhyiki.get_data("IIP", {
    "base_year": "2011-12",
    "financial_year": "2023-24",
}, format="df")

---
## GENDER - Gender Statistics

147 indicators covering demographics, health, education, labour, financial inclusion, and crimes against women.

In [ ]:
indicators = esankhyiki.get_indicators("GENDER")
print(f"Total indicators: {len(indicators)}\n")
for i in indicators[:10]:
    print(f"  {i['indicator_code']}: {i['description']}")

In [ ]:
# Sex ratio (indicator_code=2)
esankhyiki.get_data("GENDER", {"indicator_code": 2}, format="df")

---
## ENVSTATS - Environment Statistics

124 indicators covering climate, water, forests, biodiversity, pollution, and natural disasters.

In [ ]:
indicators = esankhyiki.get_indicators("ENVSTATS")
print(f"Total indicators: {len(indicators)}\n")
for i in indicators[:10]:
    print(f"  {i['indicator_code']}: {i.get('description', i.get('label', ''))}")

In [ ]:
# Mean temperature trend (indicator_code=1)
esankhyiki.get_data("ENVSTATS", {"indicator_code": 1}, format="df")

---
## RBI - Foreign Trade and Exchange Rates

39 indicators on balance of payments, forex reserves, exchange rates (155 currencies), NRI deposits.

In [ ]:
indicators = esankhyiki.get_indicators("RBI")
for i in indicators:
    print(f"  {i['indicator_code']}: {i['label']}")

In [ ]:
# Get valid sub_indicator_codes for Balance of Payments (indicator_code=22)
metadata = esankhyiki.get_metadata("RBI", sub_indicator_code=22)
metadata

In [ ]:
# Balance of Payments
esankhyiki.get_data("RBI", {"indicator_code": 22}, format="df")

---
## EC - Economic Census

District-level establishment and worker counts across three census editions (EC4/EC5/EC6).

In [ ]:
esankhyiki.get_indicators("EC", format="df")

In [ ]:
# EC6 (2013-14) - available states and filter options
metadata = esankhyiki.get_metadata("EC", indicator_code=1)
metadata

In [ ]:
# EC6 - top districts in Maharashtra by establishments
esankhyiki.get_data("EC", {
    "indicator_code": 1,
    "state": "27",       # Maharashtra
    "mode": "ranking",
    "top5opt": "2",
}, format="df")

---
## MNRE - Renewable Energy

State-wise monthly installed capacity (in MW) for solar, wind, hydro, bio, and total renewable power from the Ministry of New and Renewable Energy.

In [2]:
# Energy types: 1=Solar, 2=Wind, 3=Hydro, 4=Bio, 5=Total
# Solar/Hydro/Bio support category_code breakdown; Wind and Total do not
esankhyiki.get_indicators("MNRE", format="df")

,type_of_renewable_energy_code,label
0,1,Solar Power
1,2,Wind Power
2,3,Hydro Power
3,4,Bio Power
4,5,Total Power


In [3]:
# Filter values for Solar Power (type 1) - returns valid category_codes
metadata = esankhyiki.get_metadata("MNRE", indicator_code=1)
metadata

[{'year': [{'year': 2026},
   {'year': 2025},
   {'year': 2024},
   {'year': 2023},
   {'year': 2022},
   {'year': 2021},
   {'year': 2020}],
  'month': [{'month_code': 12, 'label': 'December'},
   {'month_code': 11, 'label': 'November'},
   {'month_code': 10, 'label': 'October'},
   {'month_code': 9, 'label': 'September'},
   {'month_code': 8, 'label': 'August'},
   {'month_code': 7, 'label': 'July'},
   {'month_code': 6, 'label': 'June'},
   {'month_code': 5, 'label': 'May'},
   {'month_code': 4, 'label': 'April'},
   {'month_code': 3, 'label': 'March'},
   {'month_code': 2, 'label': 'February'},
   {'month_code': 1, 'label': 'January'}],
  'state': [{'state_code': 36, 'label': 'All India'},
   {'state_code': 39, 'label': 'Andaman and Nicobar Islands'},
   {'state_code': 1, 'label': 'Andhra Pradesh'},
   {'state_code': 2, 'label': 'Arunachal Pradesh'},
   {'state_code': 3, 'label': 'Assam'},
   {'state_code': 4, 'label': 'Bihar'},
   {'state_code': 30, 'label': 'Chandigarh'},
   {'st

In [4]:
esankhyiki.get_data("MNRE", {
    "type_of_renewable_energy_code": 1,  # Solar Power
    "state_code": 36,                    # All India
    "year": "2023",
}, format="df")

,type_of_renewable_energy,year,month,state,category,value
0,Solar Power,2023,December,All India,Ground Mounted Solar Power,56920.20
1,Solar Power,2023,December,All India,Rooftop Solar Power,11078.94
2,Solar Power,2023,December,All India,Hybrid Solar Component,2571.96
3,Solar Power,2023,December,All India,Off-Grid Solar Power/KUSUM,2748.39
4,Solar Power,2023,December,All India,Solar Power Total,73319.50
5,Solar Power,2023,November,All India,Ground Mounted Solar Power,55958.18
6,Solar Power,2023,November,All India,Rooftop Solar Power,11078.94
7,Solar Power,2023,November,All India,Hybrid Solar Component,2570.96
8,Solar Power,2023,November,All India,Off-Grid Solar Power/KUSUM,2703.53
9,Solar Power,2023,November,All India,Solar Power Total,72311.62


---
## Output Formats

All functions support `format="dict"` (default), `format="df"` (pandas DataFrame), or `format="csv"`.

In [ ]:
csv_str = esankhyiki.get_data("PLFS", {
    "indicator_code": 3,
    "frequency_code": 1,
    "year_type_code":1,
    "year": "2023-24",
    "state_code": 99,
    "gender_code": 3,
    "age_code": 1,
    "sector_code": 3,
}, format="csv")

print(csv_str[:600])

---
## Error Handling

In [ ]:
# Typo in dataset name - fuzzy suggestion
try:
    esankhyiki.get_indicators("plfs_data")
except InvalidDatasetError as e:
    print(e)

In [ ]:
# Missing required parameter
try:
    esankhyiki.get_metadata("PLFS", frequency_code=1)
except InvalidFilterError as e:
    print(e)

In [ ]:
# Invalid filter parameter name
try:
    esankhyiki.get_data("PLFS", {
        "indicator_code": 1,
        "frequency_code": 1,
        "typo_param": "hello",
    })
except InvalidFilterError as e:
    print(e)